## DataFrame Operations

* Cover basic operations with Spark DataFrames.
* Use stock data from Walmart.

In [1]:
!curl https://raw.githubusercontent.com/markumreed/colab_pyspark/main/WMT.csv >> WMT.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 89556  100 89556    0     0   203k      0 --:--:-- --:--:-- --:--:--  203k


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('operations').getOrCreate()
df = spark.read.csv('WMT.csv', inferSchema=True, header=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/12 12:10:42 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/12 12:10:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 12:10:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [4]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: integer (nullable = true)



In [5]:
df.show(5)

+----------+---------+---------+---------+---------+---------+--------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|
+----------+---------+---------+---------+---------+---------+--------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|
|2016-01-22|62.439999|63.259998|62.130001|62.689999|55.632324| 9197500|
|2016-01-25|62.779999|    63.82|62.549999|63.450001|56.306763|12823400|
|2016-01-26|63.360001|64.470001|63.259998|     64.0|56.794834| 9441200|
+----------+---------+---------+---------+---------+---------+--------+
only showing top 5 rows


### Filtering Data

* DataFrames allow for quick filtering of data based on conditions.

In [7]:
df.filter('Close<62').show()

+----------+---------+---------+---------+---------+---------+--------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|
+----------+---------+---------+---------+---------+---------+--------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|
+----------+---------+---------+---------+---------+---------+--------+



In [8]:
df.filter('Close<62').select('Open').show()

+---------+
|     Open|
+---------+
|61.799999|
|    60.98|
+---------+



In [9]:
df.filter('Close<62').select(['Date', 'Open']).show()

+----------+---------+
|      Date|     Open|
+----------+---------+
|2016-01-20|61.799999|
|2016-01-21|    60.98|
+----------+---------+



### Using Comparison Operators

* Using comparison operators will look similar to SQL operators.
* Make to call the entire column within the dataframe

In [10]:
df.filter(df['Close']<62).show()

+----------+---------+---------+---------+---------+---------+--------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|
+----------+---------+---------+---------+---------+---------+--------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|
+----------+---------+---------+---------+---------+---------+--------+



In [15]:
df.filter((df['Close']<62) & (df['Open']>60)).show()

+----------+---------+---------+---------+---------+---------+--------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|
+----------+---------+---------+---------+---------+---------+--------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|
+----------+---------+---------+---------+---------+---------+--------+



In [16]:
df.filter(df['Open']==60.98).show()

+----------+-----+---------+-----+---------+---------+--------+
|      Date| Open|     High|  Low|    Close|Adj Close|  Volume|
+----------+-----+---------+-----+---------+---------+--------+
|2016-01-21|60.98|62.790001|60.91|61.880001|54.913509|12089200|
+----------+-----+---------+-----+---------+---------+--------+



In [17]:
df.filter(df['Open']==60.98).collect()

[Row(Date=datetime.date(2016, 1, 21), Open=60.98, High=62.790001, Low=60.91, Close=61.880001, Adj Close=54.913509, Volume=12089200)]

In [18]:
res = df.filter(df['Open']==60.98).collect()

In [20]:
type(res[0])

pyspark.sql.types.Row

In [21]:
res[0].asDict()

{'Date': datetime.date(2016, 1, 21),
 'Open': 60.98,
 'High': 62.790001,
 'Low': 60.91,
 'Close': 61.880001,
 'Adj Close': 54.913509,
 'Volume': 12089200}

In [22]:
for item in res[0]:
    print(item)

2016-01-21
60.98
62.790001
60.91
61.880001
54.913509
12089200


In [23]:
import pandas as pd

In [25]:
pd.Series(res[0].asDict())

Date         2016-01-21
Open              60.98
High          62.790001
Low               60.91
Close         61.880001
Adj Close     54.913509
Volume         12089200
dtype: object